# Introduction to Federated Learning

A hands-on tutorial demonstrating the fundamentals of federated learning through practical implementation.

## Building Blocks

### 1. Neural Network Architecture

We'll start by defining a simple neural network classifier. This model has:
- An input layer that accepts 10 features
- A hidden layer with 8 neurons and ReLU activation
- An output layer for binary classification

This architecture is intentionally simple to focus on the federated learning concepts rather than model complexity.

In [ ]:
import torch
import torch.nn as nn

class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=8, output_dim=2):
        torch.manual_seed(45)
        super(SimpleClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

### 2. Training Infrastructure

The `ClassificationTrainer` class encapsulates the training and evaluation logic:
- **Training**: Performs forward/backward passes and weight updates
- **Evaluation**: Computes accuracy on a test dataset

This abstraction separates the training mechanics from the federated learning orchestration.

In [ ]:
class ClassificationTrainer:
    def __init__(self, model, learning_rate=0.01):
        self.model = model
        self.optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
        self.criterion = nn.CrossEntropyLoss()
    
    def train(self, dataloader, epochs=1):
        self.model.train()
        for epoch in range(epochs):
            total_loss = 0
            for x, y in dataloader:
                self.optimizer.zero_grad()
                outputs = self.model(x)
                loss = self.criterion(outputs, y)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()
            avg_loss = total_loss / len(dataloader)
    
    def evaluate(self, dataloader):
        self.model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for x, y in dataloader:
                outputs = self.model(x)
                _, predicted = torch.max(outputs, 1)
                total += y.size(0)
                correct += (predicted == y).sum().item()
        accuracy = correct / total
        return accuracy

### 3. Data Preparation

We generate synthetic datasets that simulate a real-world federated learning scenario: **different data distributions across sites**.

**Datasets Generated:**
- **Client 1 Train and Test Set**
- **Client 2 Train and Test Set**
- **Centralized Test Set**
  
**The Key Difference:** By using different random seeds, each client's data comes from a slightly different distribution of the feature space. This simulates the real-world scenario where different hospitals, institutions, or regions collect data that varies in subtle but important ways.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))
from src.util import get_datasets, make_plot

datasets = get_datasets()

client_1_train = datasets['client_1_train']
client_1_test = datasets['client_1_test']

client_2_train = datasets['client_2_train']
client_2_test = datasets['client_2_test']

centralized_test = datasets['centralized_test']

## Baseline: Isolated Training

Before exploring federated learning, let's establish a baseline by training each client independently.

**Experimental setup:**
- Client 1 trains a model using only their local data for 100 epochs
- Client 2 trains a separate model using only their local data for 100 epochs

**Evaluation strategy:**
\
We'll evaluate each model on both test sets to understand generalization
1. Its own test set (in-distribution data)
2. The other client's test set (out-of-distribution data)

This will reveal the limitations of isolated training when data distributions vary across sites.

In [ ]:
# Create a model on client 1 and train for 100 epochs on client 1's train set
client_1_model = ClassificationTrainer(model=SimpleClassifier())
client_1_model.train(dataloader=client_1_train, epochs=100)

# Create a model on client 2 and train for 100 epochs on client 2's train set
client_2_model = ClassificationTrainer(model=SimpleClassifier())
client_2_model.train(dataloader=client_2_train, epochs=100)

# Evaluate each model on their own test set as well as the other client's test set
results = []
results.append(client_1_model.evaluate(dataloader=client_1_test))
results.append(client_2_model.evaluate(dataloader=client_2_test))
results.append(client_1_model.evaluate(dataloader=client_2_test))
results.append(client_2_model.evaluate(dataloader=client_1_test))

# Plot results
make_plot(results=results)

## Challenge: Isolated Training

### Results Analysis

The confusion matrix above reveals a critical problem with isolated training.

**Key Observations:**
- **Own test set**: Each model achieves ~90% accuracy on its own test data
- **Other client's test set**: Performance drops to ~50% (random guessing)

**The Problem:** Each client's model has learned patterns specific to *only* their local data distribution, failing to generalize to other clients' data.

## Solution: Federated Learning

**This is the exact problem federated learning solves.** By enabling collaboration without data sharing, federated learning allows models to learn from diverse data distributions while maintaining privacy.

### 4. Client Abstraction

The `Client` class encapsulates:
- Local model and datasets
- Training and evaluation capabilities

Each client maintains data privacy while participating in collaborative learning.

In [ ]:
from typing import Optional

class Client:
    def __init__(self, model, train_dataset, test_dataset):
        # Initialize model and train/test set on client
        self.model = model
        self.train_dataset = train_dataset
        self.test_dataset = test_dataset

        # Initialize trainer on client
        self.trainer = ClassificationTrainer(self.model)
    
    def train(self):
        # Train for one epoch on internal train set
        self.trainer.train(self.train_dataset, epochs=1)
    
    def evaluate(self, dataset: Optional[torch.utils.data.DataLoader] = None):
        # Run evaluation on provided dataset or internal test set if none provided
        if dataset:
            return self.trainer.evaluate(dataset)
        else:
            return self.trainer.evaluate(self.test_dataset)

### 5. Server Orchestration

The `Server` class manages the federated learning process:
- **Aggregation**: Combines client models using weighted averaging (FedAvg algorithm)
- **Distribution**: Receives local client weights and sends the global model back to clients

**Key Insight:** Only model parameters are shared, preserving data privacy.

In [ ]:
class Server:
    def __init__(self, model):
        # Initialize model on server (global model)
        self.model = model
    
    def receive_models(self, results):
        # Store weights sent by clients on server
        self.results = results
    
    def aggregate_weights(self):
        # When all weights have been received, aggregate them
        with torch.no_grad():
            # Iterate through each parameter of the global model
            for k, v in self.model.named_parameters():
                # Zero out the weights
                v.zero_()
                for result in self.results:
                    # Add the weights of the parameter from each client divided by the number of clients
                    v.add_(result[k], alpha=(1/len(self.results)))

### 6. Federated Training Loop

The federated training process follows a structured cycle:

**Each Round:**
1. **Local Training**: Each client trains on their private data for one epoch
2. **Upload**: Clients send model weights to the server
3. **Aggregation**: Server computes weighted average of client models
4. **Download**: Clients receive updated global model
5. **Repeat**: Process continues for the number of rounds defined

This iterative approach allows the global model to learn from all data distributions without centralizing the data.

In [ ]:
# Create client 1
client_1 = Client(
    model=SimpleClassifier(),
    train_dataset=client_1_train,
    test_dataset=client_1_test
)

# Create client 2
client_2 = Client(
    model=SimpleClassifier(),
    train_dataset=client_2_train,
    test_dataset=client_2_test
)

# Create server
server = Server(
    model=SimpleClassifier()
)

# Specify number of federated training rounds to run
num_rounds = 100

for round in range(num_rounds):
    # Each client trains for 1 epoch
    client_1.train()
    client_2.train()

    # Each client sends their set of weights to the server
    client_1_weights = client_1.model.state_dict()
    client_2_weights = client_2.model.state_dict()
    server.receive_models(results=[client_1_weights, client_2_weights])
    
    # The server aggregates the weights
    server.aggregate_weights()
    
    # The server sends the weights back to the clients
    global_weights = server.model.state_dict()
    client_1.model.load_state_dict(global_weights)
    client_2.model.load_state_dict(global_weights)


## Results: Federated vs. Isolated Training

### Performance Comparison

Let's evaluate:
1. Global model performance on each client's test set
2. Individual models on the centralized test set
3. Global model on the centralized test set

In [ ]:
# Evaluate global model on individual client test sets
results.append(client_1.evaluate())
results.append(client_2.evaluate())

# Evaluate individual client models on centralized test set
results.append(client_1_model.evaluate(dataloader=centralized_test))
results.append(client_2_model.evaluate(dataloader=centralized_test))

# Evaluate global model on centralized test set
results.append(client_1.evaluate(dataset=centralized_test))

# Plot the results
make_plot(results=results)

### Impact of Federated Learning

**Analysis:**
- **Generalization**: The global model significantly outperforms isolated models on unseen data
- **Local Performance**: Federated learning maintains (or slightly improves) performance on each client's test set
- **Real-World Readiness**: The dramatic improvement on centralized data indicates better generalization

**Privacy-Utility Trade-off:** Federated learning achieves near-optimal performance while keeping data decentralized.

### Understanding the Results

The centralized test set performance reveals the true value of federated learning. Without it, sites observe only a modest 0.5% improvement, but this underestimates the actual benefit. What if we didn't have a centralized test set? Well you may notice that all performance on the centralized test set is just the models average performance on the individual client test sets. Knowing this, we can start to abstract the idea of federated learning to federated analytics and computing in general!

## Next Steps

### Moving to Production

You now understand the core concepts of federated learning:
- ✅ Client-server architecture
- ✅ Federated averaging algorithm
- ✅ Privacy-preserving collaborative training

### What's Missing?

**Real-world deployment requires:**
1. **Secure Communication**: Sites and servers need secure ways to exchange weights
2. **Framework Integration**: Production-ready tools for managing federated workflows
3. **Scalability**: Support for many clients and large models

### Coming Up
The next tutorial demonstrates how to implement this same federated learning workflow using **NVFlare**, an industry-standard framework that handles:
- Secure communication between clients and servers
- Job configuration and workflow management  
- Simulation for local testing before production deployment

**Continue to:** `NVFlare_Simulation`
```
